# mixLSTM Test Notebook

This notebook demonstrates how to create a small sample dataset, initialize `mixLSTM`, and run a forward pass.

## 1. Environment Setup

In [1]:
import random
import numpy as np
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.datasets.splitter import split_by_sample

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cpu


## 2. Create Sample Dataset

We create synthetic time-series samples with shape `(T, input_dim)` stored under the input key `series`.

In [2]:
# Dataset parameters
num_samples = 200
T = 50  # sequence length
input_dim = 3
n_classes = 5

samples = [
    {
        "patient_id": f"patient-{i}",
        "visit_id": "visit-0",
        "series": torch.randn(T, input_dim).numpy().tolist(),
        "label": int(i % n_classes),
    }
    for i in range(num_samples)
]

input_schema = {"series": "tensor"}
output_schema = {"label": "multiclass"}

dataset = create_sample_dataset(
    samples=samples,
    input_schema=input_schema,
    output_schema=output_schema,
    dataset_name="mixlstm_demo",
)

print(f"Created dataset with {len(dataset)} samples")
print(f"Input schema: {dataset.input_schema}")
print(f"Output schema: {dataset.output_schema}")

Label label vocab: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}


Created dataset with 200 samples
Input schema: {'series': 'tensor'}
Output schema: {'label': 'multiclass'}


## 3. Split Dataset

In [3]:
train_data, val_data, test_data = split_by_sample(dataset, [0.7, 0.15, 0.15], seed=SEED)

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")
print(f"Test: {len(test_data)} samples")

train_loader = get_dataloader(train_data, batch_size=8, shuffle=True)
val_loader = get_dataloader(val_data, batch_size=8, shuffle=False)
test_loader = get_dataloader(test_data, batch_size=8, shuffle=False)

Train: 140 samples
Val: 30 samples
Test: 30 samples


## 4. Initialize `mixLSTM` Model

In [4]:
from pyhealth.models import MixLSTM

model = MixLSTM(dataset=dataset, num_experts=10, hidden_size=64)
model = model.to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

Model created with 180390 parameters


## 5. Test Forward Pass

In [5]:
# Fetch a batch and run a forward pass
batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(**batch)

print("Output keys:", outputs.keys())
print(f"Loss: {outputs['loss'].item():.4f}")
print(f"Logits shape: {outputs['logit'].shape}")

Output keys: dict_keys(['logit', 'y_prob', 'loss', 'y_true'])
Loss: 1.6047
Logits shape: torch.Size([8, 5])


## 7. Notes

- Adjust `input_dim`, `T`, and model hyperparameters to match your real dataset.
- `mixLSTM` expects input tensors of shape `(batch, seq_len, input_dim)`.
- If you encounter device mismatches, ensure tensors and model are on the same `device`.